### **Product Review Analysis Workflow**

In [2]:
import os
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate

os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")
model = ChatGoogleGenerativeAI(
    model = "gemini-3.6-flash"
)


print(f"[INFO] Model : {model.model} Loaded Successfully!")

[INFO] Model : gemini-3.6-flash Loaded Successfully!


### **Defining SentimentSchema and Diagnosis Schema**

In [4]:
from pydantic import BaseModel, Field
from typing import Literal, TypedDict

class SentimentSchema(BaseModel):
    sentiment : Literal['positive','negative'] = Field(description="sentiment of the review")


class DiagnosisSchema(BaseModel):
    issue_type : Literal['UX', 'Performance', 'Bug','Other'] = Field(description="Category of the issue mentioned in the review!")
    tone : Literal['angry','frustrated','calm','staisfied'] = Field(description="Emotional Tone expresseed by the reviewer!")
    urgency: Literal['low','medium','high'] = Field(description="How Urgent or critical the issue appears to be!")

### **Creating thr Review State**

In [ ]:
class ReviewState(TypedDict):
    review : str
    sentiment : Literal['positive','negative']
    diagnosis : dict
    response : str

### **Node 1 : Find Sentiment**  

In [6]:
sentiment_model = model.with_structured_output(SentimentSchema)

def find_sentiment(state:ReviewState)->Literal['positive','negative']:
    review = state['review']
    response = sentiment_model.invoke(f"Find out the sentiment of the review - {review}")
    return {
        "sentiment" : response.sentiment
    }


### **Node 2 : Positive Response**

In [8]:
def positive_response(state:ReviewState):
    review = state['review']
    prompt = f"""
    Write a warm thank you message in response to this review.\n
    {review}.\n 
    Also kindly ask the user to leave a feedback on our website.
    """
    response = model.invoke(prompt)
    return {
        "response" : response
    }

### **Node 3 : Run Diagnosis**

In [10]:
diagnose_model = model.with_structured_output(DiagnosisSchema)
def run_diagnosis(state:ReviewState):
    review = state['review']
    prompt = f"""
        diagnose the following negative review.\n
        {review}.\n 
        Return issue_type, tone, urgency.
    """
    diagnose = diagnose_model.invoke(prompt)
    return {
        "diagnosis" : diagnose.model_dump()
    }


### **Node 4 : Negative Response**

In [11]:
def negative_response(state:ReviewState):
    diagnosis = state['diagnosis']
    prompt = f"""
    You are a support assitant.\n
    The user had '{diagnosis['issue_type']}' issue, sounded '{diagnosis['tone']}' and marked urgency as '{diagnosis['urgency']}'.\n
    write an empathetic, helpful resolution message.
    """

    response = model.invoke(prompt).content[0]['text']
    return {
        "response" : response
    }

### **Create the Condition Check Edge**

In [12]:
def condition_check(state:ReviewState)->Literal["positive_response","negative_response"]:
    sentiment = state['sentiment'].lower()
    if sentiment == "positive":
        return "positive_response"
    else:
        return "run_diagnosis"

### **Creating the State graph**

In [ ]:
# from langgraph.graph import StateGraph,START, END

# graph = StateGraph(ReviewState)

# graph.add_node('find_sentiment', find_sentiment)
# graph.add_node('positive_response', positive_response)
# graph.add_node('run_diagnosis', run_diagnosis)
# graph.add_node('negative_response',negative_response)

# graph.add_edge(START, 'find_sentiment')
# graph.add_conditional_edges('find_sentiment', condition_check)
# graph.add_edge('run_diagnosis', 'negative_response')
# graph.add_edge('positive_response', END)
# graph.add_edge('negative_response', END)
